# Notebook 01: CGA Feature Extraction

Extracts all utterance-level and conversation-level features from the
Conversations Gone Awry (CGA) corpus for downstream AMD and CSD analysis.

Outputs:
- Utterance-level parquet with GoEmotions distributions, DA distributions,
  toxicity scores, VADER sentiment, NRC VAD, context variables, and metadata.
- Conversation-level summary parquet with anchors, AMD values, and repair proxies.

In [ ]:
!pip install -q convokit transformers vaderSentiment torch scikit-learn pandas pyarrow tqdm nrclex nltk textblob
!python -m textblob.download_corpora

In [ ]:
import os
import pickle
import warnings
from collections import Counter, defaultdict
from pathlib import Path

import nltk
import numpy as np
import pandas as pd
import torch
from sklearn.cluster import KMeans
from sklearn.feature_extraction.text import TfidfVectorizer
from tqdm import tqdm
from transformers import AutoModelForSequenceClassification, AutoTokenizer, pipeline
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer

warnings.filterwarnings("ignore")
nltk.download("stopwords", quiet=True)
nltk.download("punkt", quiet=True)
nltk.download("punkt_tab", quiet=True)

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {DEVICE}")

## 1. Load CGA Corpus and Filter Conversations

In [ ]:
from convokit import Corpus, download

corpus = Corpus(filename=download("conversations-gone-awry-corpus"))

conversations = []
for convo_id, convo in corpus.conversations.items():
    utterance_list = list(convo.iter_utterances())
    if len(utterance_list) >= 10:
        has_attack = convo.meta.get("conversation_has_personal_attack", False)
        conversations.append({
            "convo_id": convo_id,
            "utterances": utterance_list,
            "derails": has_attack,
            "n_turns": len(utterance_list),
        })

N_total = len(conversations)
N_derail = sum(1 for c in conversations if c["derails"])
N_civil = N_total - N_derail

print(f"N_total: {N_total}")
print(f"N_derail: {N_derail}")
print(f"N_civil: {N_civil}")

## 2. Initialize Models

In [ ]:
GOEMOTIONS_MODEL = "SamLowe/roberta-base-go_emotions"
goemotions_tokenizer = AutoTokenizer.from_pretrained(GOEMOTIONS_MODEL)
goemotions_model = AutoModelForSequenceClassification.from_pretrained(GOEMOTIONS_MODEL).to(DEVICE)
goemotions_model.eval()

GOEMOTIONS_LABELS = [
    "admiration", "amusement", "anger", "annoyance", "approval",
    "caring", "confusion", "curiosity", "desire", "disappointment",
    "disapproval", "disgust", "embarrassment", "excitement", "fear",
    "gratitude", "grief", "joy", "love", "nervousness",
    "optimism", "pride", "realization", "relief", "remorse",
    "sadness", "surprise", "neutral",
]

print(f"GoEmotions model loaded: {len(GOEMOTIONS_LABELS)} emotion classes")

In [ ]:
TOXICITY_MODEL = "unitary/toxic-bert"
toxicity_pipeline = pipeline(
    "text-classification",
    model=TOXICITY_MODEL,
    tokenizer=TOXICITY_MODEL,
    device=0 if DEVICE == "cuda" else -1,
    truncation=True,
    max_length=512,
)

vader_analyzer = SentimentIntensityAnalyzer()

print("Toxicity and VADER models loaded")

In [ ]:
from nrclex import NRCLex


def get_nrc_vad(text):
    """Extract average valence from NRC VAD lexicon for words in text."""
    emotion_obj = NRCLex(text)
    scores = emotion_obj.raw_emotion_scores
    positive = scores.get("positive", 0)
    negative = scores.get("negative", 0)
    total = positive + negative
    if total == 0:
        return 0.0
    return (positive - negative) / total


print("NRC lexicon loaded")

## 3. Extract Utterance-Level Features

In [ ]:
def extract_goemotions_batch(texts, batch_size=32):
    """Extract GoEmotions probability distributions for a batch of texts."""
    all_distributions = []

    for start_idx in range(0, len(texts), batch_size):
        batch_texts = texts[start_idx:start_idx + batch_size]
        encoded = goemotions_tokenizer(
            batch_texts,
            padding=True,
            truncation=True,
            max_length=512,
            return_tensors="pt",
        ).to(DEVICE)

        with torch.no_grad():
            logits = goemotions_model(**encoded).logits
            probabilities = torch.sigmoid(logits)
            row_sums = probabilities.sum(dim=1, keepdim=True)
            row_sums = torch.clamp(row_sums, min=1e-8)
            normalized = probabilities / row_sums

        all_distributions.extend(normalized.cpu().numpy().tolist())

    return all_distributions


def extract_toxicity_batch(texts, batch_size=32):
    """Extract toxicity probabilities for a batch of texts."""
    scores = []
    for start_idx in range(0, len(texts), batch_size):
        batch_texts = texts[start_idx:start_idx + batch_size]
        results = toxicity_pipeline(batch_texts)
        for result in results:
            if result["label"] == "toxic":
                scores.append(result["score"])
            else:
                scores.append(1.0 - result["score"])
    return scores


print("Batch extraction functions defined")

In [ ]:
all_utterance_records = []

for convo_data in tqdm(conversations, desc="Processing conversations"):
    convo_id = convo_data["convo_id"]
    derails = convo_data["derails"]
    utterances = convo_data["utterances"]

    texts = []
    speakers = []
    for utt in utterances:
        text = utt.text if utt.text else ""
        texts.append(text[:512])
        speakers.append(utt.speaker.id if utt.speaker else "unknown")

    goemotions_dists = extract_goemotions_batch(texts)
    toxicity_scores = extract_toxicity_batch(texts)

    vader_scores = [vader_analyzer.polarity_scores(t)["compound"] for t in texts]
    nrc_vad_scores = [get_nrc_vad(t) for t in texts]

    attack_turn = None
    for turn_idx, utt in enumerate(utterances):
        if utt.meta.get("comment_has_personal_attack", False):
            attack_turn = turn_idx
            break

    for turn_idx in range(len(utterances)):
        record = {
            "convo_id": convo_id,
            "turn_idx": turn_idx,
            "speaker_id": speakers[turn_idx],
            "text": texts[turn_idx],
            "derails": derails,
            "attack_turn": attack_turn,
            "goemotions_dist": goemotions_dists[turn_idx],
            "toxicity_score": toxicity_scores[turn_idx],
            "vader_compound": vader_scores[turn_idx],
            "nrc_vad": nrc_vad_scores[turn_idx],
        }
        all_utterance_records.append(record)

print(f"Total utterance records: {len(all_utterance_records)}")

## 4. Topic Clustering and Context Variable Assignment

In [ ]:
TOPIC_K = 5

convo_grouped = defaultdict(list)
for record in all_utterance_records:
    convo_grouped[record["convo_id"]].append(record)

for convo_id, records in tqdm(convo_grouped.items(), desc="Topic clustering"):
    texts_in_convo = [r["text"] for r in records]

    effective_k = min(TOPIC_K, len(texts_in_convo))
    if effective_k < 2:
        for r in records:
            r["topic_cluster"] = 0
        continue

    vectorizer = TfidfVectorizer(max_features=1000, stop_words="english")
    tfidf_matrix = vectorizer.fit_transform(texts_in_convo)

    kmeans = KMeans(n_clusters=effective_k, random_state=RANDOM_SEED, n_init=10)
    cluster_labels = kmeans.fit_predict(tfidf_matrix)

    for idx, r in enumerate(records):
        r["topic_cluster"] = int(cluster_labels[idx])

print("Topic clustering complete")

In [ ]:
SWDA_INT_TO_TAG = {
    0: "sd", 1: "b", 3: "aa", 4: "%", 6: "qy", 7: "x", 8: "ny", 9: "fc",
    10: "qw", 11: "nn", 12: "bk", 13: "h", 14: "qy^d", 16: "^q", 17: "bf",
    18: 'fo_o_fw_"_by_bc', 19: "na", 20: "ad", 21: "^2", 22: "b^m",
    23: "qo", 24: "qh", 25: "^h", 26: "ar", 27: "ng", 28: "br", 29: "no",
    30: "fp", 31: "qrr", 32: "arp_nd", 33: "t3", 34: "oo_co_cc", 35: "aap_am",
    36: "t1", 37: "bd", 38: "^g", 39: "qw^d", 40: "fa", 41: "ft", 42: "+",
    43: "x", 44: 'fo_o_fw_"_by_bc', 45: "ba",
}

DA_MODEL_PATH = Path("/content/drive/MyDrive/phase-transition-amd/models/da_classifier")

USE_CUSTOM_DA = False
USE_PRETRAINED_DA = False

if DA_MODEL_PATH.exists() and (DA_MODEL_PATH / "model.safetensors").exists():
    import json as _json
    da_tokenizer = AutoTokenizer.from_pretrained(str(DA_MODEL_PATH))
    da_model = AutoModelForSequenceClassification.from_pretrained(str(DA_MODEL_PATH)).to(DEVICE)
    da_model.eval()

    with open(DA_MODEL_PATH / "metadata.json") as f:
        da_metadata = _json.load(f)
    DA_RAW_LABELS = da_metadata["label_names"] 
    DA_NUM_LABELS = da_metadata["num_labels"]

    DA_TAG_NAMES = []
    for raw_label in DA_RAW_LABELS:
        int_label = int(raw_label) if isinstance(raw_label, (int, float, str)) else raw_label
        tag = SWDA_INT_TO_TAG.get(int_label, f"unk_{int_label}")
        DA_TAG_NAMES.append(tag)

    USE_CUSTOM_DA = True
    print(f"Loaded fine-tuned DA classifier from {DA_MODEL_PATH}")
    print(f"  sw_acc: {da_metadata['sw_acc']}, num_labels: {DA_NUM_LABELS}")
    print(f"  First 10 tag names: {DA_TAG_NAMES[:10]}")

    for tag in ["aa", "bk", "br", "ba"]:
        if tag in DA_TAG_NAMES:
            print(f"  Conciliatory tag '{tag}' -> model index {DA_TAG_NAMES.index(tag)}")
        else:
            print(f"  Conciliatory tag '{tag}' -> NOT in model outputs")
else:
    try:
        da_pipeline = pipeline(
            "text-classification",
            model="diwank/silicone-deberta-pair",
            device=0 if DEVICE == "cuda" else -1,
            truncation=True,
            max_length=512,
            top_k=None,
        )
        USE_PRETRAINED_DA = True
        print("Using pretrained DA classifier: diwank/silicone-deberta-pair")
    except Exception:
        print("WARNING: No DA classifier available; using placeholder DA labels")
        print(f"  Looked for fine-tuned model at: {DA_MODEL_PATH}")
        print("  To fix: run notebook 03 first, then copy model to Google Drive")

In [ ]:
def predict_da_custom(text):
    """Run DA classification using your fine-tuned model, returning SWDA tag names."""
    inputs = da_tokenizer(
        text[:512], truncation=True, max_length=128,
        padding="max_length", return_tensors="pt"
    ).to(DEVICE)
    with torch.no_grad():
        logits = da_model(**inputs).logits
        probs = torch.softmax(logits, dim=-1)[0].cpu().numpy()
    da_probs = {DA_TAG_NAMES[i]: float(probs[i]) for i in range(len(DA_TAG_NAMES))}
    da_label = DA_TAG_NAMES[int(probs.argmax())]
    return da_probs, da_label


for convo_id, records in tqdm(convo_grouped.items(), desc="Assigning context variables"):
    for idx, r in enumerate(records):
        if idx == 0:
            preceding_da = "none"
        else:
            preceding_da = records[idx - 1].get("da_label", "sd")

        if USE_CUSTOM_DA:
            try:
                da_probs, da_label = predict_da_custom(r["text"])
            except Exception:
                da_probs = {"sd": 0.5, "b": 0.2, "aa": 0.1, "ba": 0.1, "bk": 0.1}
                da_label = "sd"
        elif USE_PRETRAINED_DA:
            try:
                da_result = da_pipeline(r["text"][:512])
                if isinstance(da_result, list) and isinstance(da_result[0], list):
                    da_result = da_result[0]
                da_probs = {item["label"]: item["score"] for item in da_result}
                da_label = max(da_probs, key=da_probs.get)
            except Exception:
                da_probs = {"sd": 0.5, "b": 0.2, "aa": 0.1, "ba": 0.1, "bk": 0.1}
                da_label = "sd"
        else:
            da_probs = {"sd": 0.5, "b": 0.2, "aa": 0.1, "ba": 0.1, "bk": 0.1}
            da_label = "sd"

        r["da_label"] = da_label
        r["da_probs"] = da_probs
        r["preceding_da"] = preceding_da
        r["context"] = f"{preceding_da}__topic_{r['topic_cluster']}"

print("Context variable assignment complete")
if USE_CUSTOM_DA:
    print("  Used: fine-tuned RoBERTa DA classifier from Google Drive")
    sample = list(convo_grouped.values())[0][0]
    sample_probs = sample.get("da_probs", {})
    sample_q = sum(sample_probs.get(t, 0.0) for t in ["aa", "bk", "br", "ba"])
    print(f"  Sample q_da: {sample_q:.4f} (from {list(sample_probs.keys())[:5]}...)")
elif USE_PRETRAINED_DA:
    print("  Used: pretrained HF DA classifier")
else:
    print("  Used: PLACEHOLDER values (results will have null q_DA)")

## 5. Anchor Extraction and AMD Computation

In [ ]:
import re
from nltk.corpus import stopwords

STOPWORDS = set(stopwords.words("english"))
WORD_PATTERN = re.compile(r"\b[a-z]{2,}\b")
MIN_ANCHOR_FREQ = 3
MIN_PER_CELL = 3


def extract_content_words(text):
    """Extract content words from text."""
    words = WORD_PATTERN.findall(text.lower())
    return [w for w in words if w not in STOPWORDS]


def total_variation(p, q):
    """Total variation distance."""
    return 0.5 * np.sum(np.abs(np.asarray(p) - np.asarray(q)))


print(f"Anchor extraction configured: min_freq={MIN_ANCHOR_FREQ}, min_per_cell={MIN_PER_CELL}")

In [ ]:
conversation_summaries = []

for convo_id, records in tqdm(convo_grouped.items(), desc="Computing AMD per conversation"):
    speaker_ids = list(set(r["speaker_id"] for r in records))
    if len(speaker_ids) < 2:
        continue

    spk1, spk2 = speaker_ids[0], speaker_ids[1]
    texts_spk1 = [r["text"] for r in records if r["speaker_id"] == spk1]
    texts_spk2 = [r["text"] for r in records if r["speaker_id"] == spk2]

    words_spk1 = []
    for t in texts_spk1:
        words_spk1.extend(extract_content_words(t))
    words_spk2 = []
    for t in texts_spk2:
        words_spk2.extend(extract_content_words(t))

    counter1 = Counter(words_spk1)
    counter2 = Counter(words_spk2)
    shared = set(counter1.keys()) & set(counter2.keys())
    anchors = sorted([w for w in shared if counter1[w] + counter2[w] >= MIN_ANCHOR_FREQ])

    if not anchors:
        continue

    anchor_map = defaultdict(lambda: defaultdict(list))
    for r in records:
        text_words = set(WORD_PATTERN.findall(r["text"].lower()))
        for anchor in anchors:
            if anchor in text_words:
                anchor_map[(anchor, r["speaker_id"])][r["context"]].append(
                    np.array(r["goemotions_dist"])
                )

    convo_d_marg_values = []
    convo_d_cond_values = []
    convo_d_ctx_values = []

    for anchor in anchors:
        ctx_dists_1 = anchor_map.get((anchor, spk1), {})
        ctx_dists_2 = anchor_map.get((anchor, spk2), {})

        ed1 = {}
        ed2 = {}
        cnt1 = {}
        cnt2 = {}

        for ctx, dists in ctx_dists_1.items():
            if len(dists) >= MIN_PER_CELL:
                ed1[ctx] = np.mean(dists, axis=0)
                cnt1[ctx] = len(dists)

        for ctx, dists in ctx_dists_2.items():
            if len(dists) >= MIN_PER_CELL:
                ed2[ctx] = np.mean(dists, axis=0)
                cnt2[ctx] = len(dists)

        if not ed1 or not ed2:
            continue

        t1 = sum(cnt1.values())
        t2 = sum(cnt2.values())
        cw1 = {c: n / t1 for c, n in cnt1.items()}
        cw2 = {c: n / t2 for c, n in cnt2.items()}

        all_ctx = set(cw1.keys()) | set(cw2.keys())
        n_emo = len(GOEMOTIONS_LABELS)
        marg1 = sum(cw1.get(c, 0) * ed1.get(c, np.zeros(n_emo)) for c in all_ctx)
        marg2 = sum(cw2.get(c, 0) * ed2.get(c, np.zeros(n_emo)) for c in all_ctx)
        d_marg = total_variation(marg1, marg2)

        shared_ctx = set(ed1.keys()) & set(ed2.keys())
        if shared_ctx:
            wt_sum = 0.0
            tv_sum = 0.0
            for c in shared_ctx:
                w = cnt1[c] + cnt2[c]
                tv_sum += w * total_variation(ed1[c], ed2[c])
                wt_sum += w
            d_cond = tv_sum / wt_sum if wt_sum > 0 else np.nan
        else:
            d_cond = np.nan

        p1 = np.array([cw1.get(c, 0) for c in all_ctx])
        p2 = np.array([cw2.get(c, 0) for c in all_ctx])
        if p1.sum() > 0:
            p1 = p1 / p1.sum()
        if p2.sum() > 0:
            p2 = p2 / p2.sum()
        d_ctx = total_variation(p1, p2)

        convo_d_marg_values.append(d_marg)
        convo_d_cond_values.append(d_cond)
        convo_d_ctx_values.append(d_ctx)

    derails = records[0]["derails"]
    attack_turn = records[0].get("attack_turn", None)

    summary = {
        "convo_id": convo_id,
        "derails": derails,
        "attack_turn": attack_turn,
        "n_turns": len(records),
        "n_anchors": len(anchors),
        "mean_d_marg": float(np.nanmean(convo_d_marg_values)) if convo_d_marg_values else np.nan,
        "mean_d_cond": float(np.nanmean(convo_d_cond_values)) if convo_d_cond_values else np.nan,
        "mean_d_ctx": float(np.nanmean(convo_d_ctx_values)) if convo_d_ctx_values else np.nan,
    }
    conversation_summaries.append(summary)

print(f"Conversations with valid anchors: {len(conversation_summaries)}")

## 6. Compute Repair Proxies Per Turn

In [ ]:
CONCILIATORY_TAGS = {"aa", "bk", "br", "ba"}
EMA_ALPHA = 0.3

REPAIR_PATTERNS_COMPILED = [
    re.compile(p, re.IGNORECASE) for p in [
        r"\bwhat\b.*\?", r"\bhuh\b", r"\bsorry\b", r"\bpardon\b",
        r"\bexcuse me\b", r"\bI mean\b", r"\bactually\b", r"\bno\s*,",
        r"\bwait\b", r"\bhold on\b", r"\byeah\b", r"\bright\b",
        r"\bokay\b", r"\bmhm\b", r"\buh huh\b", r"\bI see\b",
    ]
]

for convo_id, records in tqdm(convo_grouped.items(), desc="Computing repair proxies"):
    texts_in_order = [r["text"] for r in records]

    raw_rm = np.zeros(len(records))
    for idx, text in enumerate(texts_in_order):
        for pattern in REPAIR_PATTERNS_COMPILED:
            if pattern.search(text):
                raw_rm[idx] = 1.0
                break

    smoothed_rm = np.zeros(len(records))
    if len(records) > 0:
        smoothed_rm[0] = raw_rm[0]
        for idx in range(1, len(records)):
            smoothed_rm[idx] = EMA_ALPHA * raw_rm[idx] + (1 - EMA_ALPHA) * smoothed_rm[idx - 1]

    for idx, r in enumerate(records):
        da_probs = r.get("da_probs", {})
        q_da = sum(da_probs.get(tag, 0.0) for tag in CONCILIATORY_TAGS)
        q_ce = 1.0 - r["toxicity_score"]

        r["q_da"] = q_da
        r["q_rm"] = smoothed_rm[idx]
        r["q_ce"] = q_ce

print("Repair proxy computation complete")

## 7. Save Features to Parquet

In [ ]:
OUTPUT_DIR = Path("../data")
OUTPUT_DIR.mkdir(exist_ok=True)

flat_records = []
for convo_id, records in convo_grouped.items():
    for r in records:
        flat_record = {
            "convo_id": r["convo_id"],
            "turn_idx": r["turn_idx"],
            "speaker_id": r["speaker_id"],
            "text": r["text"],
            "derails": r["derails"],
            "attack_turn": r.get("attack_turn"),
            "toxicity_score": r["toxicity_score"],
            "vader_compound": r["vader_compound"],
            "nrc_vad": r["nrc_vad"],
            "da_label": r.get("da_label", ""),
            "preceding_da": r.get("preceding_da", ""),
            "topic_cluster": r.get("topic_cluster", 0),
            "context": r.get("context", ""),
            "q_da": r.get("q_da", 0.0),
            "q_rm": r.get("q_rm", 0.0),
            "q_ce": r.get("q_ce", 0.0),
        }
        for emo_idx, emo_name in enumerate(GOEMOTIONS_LABELS):
            flat_record[f"ge_{emo_name}"] = r["goemotions_dist"][emo_idx]

        da_probs = r.get("da_probs", {})
        for da_key, da_val in da_probs.items():
            flat_record[f"da_prob_{da_key}"] = da_val

        flat_records.append(flat_record)

utterance_df = pd.DataFrame(flat_records)
utterance_df.to_parquet(OUTPUT_DIR / "cga_features.parquet", index=False)
print(f"Utterance features saved: {utterance_df.shape}")

summary_df = pd.DataFrame(conversation_summaries)
summary_df.to_parquet(OUTPUT_DIR / "cga_conversation_summary.parquet", index=False)
print(f"Conversation summaries saved: {summary_df.shape}")

print(f"\nN_total: {N_total}")
print(f"N_derail: {N_derail}")
print(f"N_civil: {N_civil}")

## [Colab only] Save outputs to Google Drive
Run this cell to mount Google Drive and copy the parquet files there for download.
**Remove this cell before submitting.**

In [ ]:
# === [Colab only] Save to Google Drive — remove before submitting ===
from google.colab import drive
drive.mount("/content/drive")

import shutil
from pathlib import Path

DRIVE_DEST = Path("/content/drive/MyDrive/phase-transition-amd/data")
DRIVE_DEST.mkdir(parents=True, exist_ok=True)

LOCAL_DATA = Path("../data")
for f in LOCAL_DATA.glob("cga_*.parquet"):
    shutil.copy2(f, DRIVE_DEST / f.name)
    print(f"Copied {f.name} -> {DRIVE_DEST / f.name}")

print(f"\nAll CGA outputs saved to Google Drive at:\n  {DRIVE_DEST}")